In [1]:
"""
Compute and store sentence embeddings for all Neo4j nodes.

What it does:
  1. Pulls every node's (name + docstring) from Neo4j
  2. Encodes with all-MiniLM-L6-v2  →  384-dim float vectors
  3. Writes them back as n.embedding on each node
  4. Prints a progress bar and final stats

"""

import logging
import time
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

/home/rahul/code/tdl_proj/Structural-Coder/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# Config — change these to match your setup 
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "pytorch2026"         
MODEL_NAME     = "all-MiniLM-L6-v2"      # 384-dim, ~80MB, fast on CPU
BATCH_SIZE     = 256                      # nodes encoded per batch
WRITE_BATCH    = 100                      # nodes written per Neo4j transaction


def fetch_all_nodes(session) -> list[dict]:
    """Pull id, name, docstring for every node in the graph."""
    result = session.run(
        """
        MATCH (n)
        RETURN elementId(n) AS eid,
               coalesce(n.qualified, n.name, '') AS name,
               coalesce(n.docstring, n.description, n.text, n.note, '') AS doc
        """
    )
    return [{"eid": r["eid"], "name": r["name"], "doc": r["doc"]}
            for r in result]


def build_text(record: dict) -> str:
    """Combine name + docstring into one input string for the encoder."""
    name = (record["name"] or "").strip()
    doc  = (record["doc"]  or "").strip()[:300]   # cap docstring at 300 chars
    return f"{name}. {doc}" if doc else name


def write_embeddings(session, batch: list[dict]) -> None:
    """
    Write embeddings back to Neo4j in one transaction.
    Uses elementId (Neo4j 5.x) for precise node matching.
    """
    session.run(
        """
        UNWIND $rows AS row
        MATCH (n) WHERE elementId(n) = row.eid
        SET n.embedding = row.embedding
        """,
        rows=batch,
    )


def main():
    log.info("Loading sentence encoder: %s", MODEL_NAME)
    encoder = SentenceTransformer(MODEL_NAME)

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

    try:
        # Fetch all nodes 
        log.info("Fetching nodes from Neo4j …")
        with driver.session() as session:
            nodes = fetch_all_nodes(session)
        log.info("Found %d nodes to embed.", len(nodes))

        if not nodes:
            log.warning("No nodes found. Is the DB populated?")
            return

        # Encode in batches 
        texts = [build_text(n) for n in nodes]
        log.info("Encoding %d texts in batches of %d …", len(texts), BATCH_SIZE)

        t0 = time.time()
        embeddings = encoder.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,   # unit-norm → cosine sim = dot product
        )
        elapsed = time.time() - t0
        log.info(
            "Encoding done in %.1fs  (%.0f nodes/sec)",
            elapsed, len(nodes) / elapsed,
        )

        # Write back to Neo4j 
        log.info("Writing embeddings to Neo4j (write batch = %d) …", WRITE_BATCH)

        write_rows = [
            {"eid": nodes[i]["eid"], "embedding": embeddings[i].tolist()}
            for i in range(len(nodes))
        ]

        with driver.session() as session:
            for start in range(0, len(write_rows), WRITE_BATCH):
                chunk = write_rows[start : start + WRITE_BATCH]
                session.execute_write(write_embeddings, chunk)
                if (start // WRITE_BATCH) % 20 == 0:
                    log.info(
                        "  Written %d / %d nodes …",
                        min(start + WRITE_BATCH, len(write_rows)),
                        len(write_rows),
                    )

        # Verify 
        with driver.session() as session:
            count = session.run(
                "MATCH (n) WHERE n.embedding IS NOT NULL RETURN count(n) AS c"
            ).single()["c"]

        log.info("━━━ Done ━━━")
        log.info("  Nodes with embeddings : %d / %d", count, len(nodes))
        log.info("  Embedding dimension   : %d", len(embeddings[0]))
        log.info("  Model                 : %s", MODEL_NAME)

    finally:
        driver.close()


if __name__ == "__main__":
    main()

18:59:29  INFO      Loading sentence encoder: all-MiniLM-L6-v2
18:59:29  INFO      Use pytorch device_name: cuda:0
18:59:29  INFO      Load pretrained SentenceTransformer: all-MiniLM-L6-v2
18:59:29  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
18:59:29  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
18:59:30  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
18:59:30  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
18:59:30  INFO      HTTP Request: HEAD https://hugg